In [3]:
# ================================================================
#  TASK 6: ML Outperformance Classifier — PortfolioIQ HackNova 2026
# ================================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             precision_score, recall_score,
                             confusion_matrix, roc_curve)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
    print("XGBoost ✅")
except:
    HAS_XGB = False
    print("Using GradientBoosting instead")

# ── CONFIG ────────────────────────────────────────────────────
INPUT_FILE   = "task1_adj_close_wide-3.csv"
HORIZON      = 20
TRADING_DAYS = 252
RF_RATE      = 0.065

STOCKS = [
    "HDFCBANK","ICICIBANK","SBIN","KOTAKBANK","AXISBANK",
    "TCS","INFY","WIPRO","HCLTECH","TECHM",
    "SUNPHARMA","DRREDDY","CIPLA","DIVISLAB","APOLLOHOSP"
]
SECTOR_MAP = {
    "HDFCBANK":"Banking","ICICIBANK":"Banking","SBIN":"Banking",
    "KOTAKBANK":"Banking","AXISBANK":"Banking",
    "TCS":"IT","INFY":"IT","WIPRO":"IT","HCLTECH":"IT","TECHM":"IT",
    "SUNPHARMA":"Pharma","DRREDDY":"Pharma","CIPLA":"Pharma",
    "DIVISLAB":"Pharma","APOLLOHOSP":"Pharma"
}

FEATURE_COLS = [
    "ret_1d","ret_5d","ret_10d","ret_20d","ret_60d",
    "rs_5d","rs_20d",
    "vol_10d","vol_20d",
    "price_vs_sma20","price_vs_sma50","price_vs_sma200","sma50_vs_sma200",
    "rsi_14","sharpe_20d","beta_60d","drawdown_20d"
]

# ── STEP 1: Load Data ─────────────────────────────────────────
print("\n[1] Loading data...")
df = pd.read_csv(INPUT_FILE)
# FIXED: Added dayfirst=True and format='mixed' to handle European/Indian date formats
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, format='mixed')
df = df.sort_values("Date").set_index("Date")
prices = df[STOCKS].copy()
nifty  = df["NIFTY50"].copy()
print(f"    {len(df)} rows | {df.index[0].date()} to {df.index[-1].date()}")

# ── STEP 2: Feature Engineering ──────────────────────────────
print("\n[2] Engineering 17 features per stock per day...")

def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / (loss + 1e-9)
    return 100 - (100 / (1 + rs))

def compute_features(price_series, nifty_series):
    r  = price_series.pct_change()
    nr = nifty_series.pct_change()
    f  = pd.DataFrame(index=price_series.index)
    f["ret_1d"]          = r
    f["ret_5d"]          = price_series.pct_change(5)
    f["ret_10d"]         = price_series.pct_change(10)
    f["ret_20d"]         = price_series.pct_change(20)
    f["ret_60d"]         = price_series.pct_change(60)
    f["rs_5d"]           = f["ret_5d"]  - nifty_series.pct_change(5)
    f["rs_20d"]          = f["ret_20d"] - nifty_series.pct_change(20)
    f["vol_10d"]         = r.rolling(10).std() * np.sqrt(TRADING_DAYS)
    f["vol_20d"]         = r.rolling(20).std() * np.sqrt(TRADING_DAYS)
    sma20  = price_series.rolling(20).mean()
    sma50  = price_series.rolling(50).mean()
    sma200 = price_series.rolling(200).mean()
    f["price_vs_sma20"]  = (price_series - sma20)  / sma20
    f["price_vs_sma50"]  = (price_series - sma50)  / sma50
    f["price_vs_sma200"] = (price_series - sma200) / sma200
    f["sma50_vs_sma200"] = (sma50 - sma200) / sma200
    f["rsi_14"]          = compute_rsi(price_series, 14)
    f["sharpe_20d"]      = (r.rolling(20).mean()*TRADING_DAYS - RF_RATE) / \
                           (r.rolling(20).std()*np.sqrt(TRADING_DAYS) + 1e-9)
    cov_roll = r.rolling(60).cov(nr)
    var_roll = nr.rolling(60).var()
    f["beta_60d"]        = cov_roll / (var_roll + 1e-9)
    roll_max = price_series.rolling(20).max()
    f["drawdown_20d"]    = (price_series - roll_max) / roll_max
    return f

all_features = []
for stock in STOCKS:
    feat_df = compute_features(prices[stock], nifty)
    feat_df["ticker"] = stock
    feat_df["sector"] = SECTOR_MAP[stock]
    fwd_stock = prices[stock].pct_change(HORIZON).shift(-HORIZON)
    fwd_nifty = nifty.pct_change(HORIZON).shift(-HORIZON)
    feat_df["target"] = (fwd_stock > fwd_nifty).astype(int)
    all_features.append(feat_df)

full_df = pd.concat(all_features).dropna().reset_index().sort_values("Date")
print(f"    Total samples : {len(full_df)}")
print(f"    Outperform %  : {full_df['target'].mean()*100:.1f}%")

# ── STEP 3: Train/Test Split (time-based, NO leakage) ─────────
print("\n[3] Time-series split 80/20...")
split_idx  = int(len(full_df) * 0.80)
train_df   = full_df.iloc[:split_idx]
test_df    = full_df.iloc[split_idx:]
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(train_df[FEATURE_COLS].values)
X_test_sc  = scaler.transform(test_df[FEATURE_COLS].values)
y_train    = train_df["target"].values
y_test     = test_df["target"].values
print(f"    Train: {len(train_df)} | Test: {len(test_df)}")

# ── STEP 4: Train Models ──────────────────────────────────────
print("\n[4] Training models...")
if HAS_XGB:
    model1 = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
        gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
        eval_metric="logloss", random_state=42, n_jobs=-1
    )
    m1_name = "XGBoost"
else:
    model1 = GradientBoostingClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, min_samples_leaf=20, random_state=42
    )
    m1_name = "GradientBoosting"

model2 = RandomForestClassifier(
    n_estimators=500, max_depth=8, min_samples_leaf=10,
    min_samples_split=20, max_features="sqrt",
    class_weight="balanced", random_state=42, n_jobs=-1
)
m2_name = "RandomForest"

model1.fit(X_train_sc, y_train)
model2.fit(X_train_sc, y_train)
print(f"    {m1_name} ✅  |  {m2_name} ✅")

# ── STEP 5: Evaluate ──────────────────────────────────────────
print("\n[5] Evaluating on test set...")

def evaluate(model, name):
    tscv   = TimeSeriesSplit(n_splits=5)
    cv_f1  = cross_val_score(model, X_train_sc, y_train, cv=tscv,
                              scoring="f1", n_jobs=-1)
    yp     = model.predict(X_test_sc)
    yprob  = model.predict_proba(X_test_sc)[:,1]
    result = {
        "Model":        name,
        "Accuracy_%":   round(accuracy_score(y_test,yp)*100, 2),
        "Precision_%":  round(precision_score(y_test,yp,zero_division=0)*100, 2),
        "Recall_%":     round(recall_score(y_test,yp,zero_division=0)*100, 2),
        "F1_%":         round(f1_score(y_test,yp,zero_division=0)*100, 2),
        "ROC_AUC":      round(roc_auc_score(y_test,yprob), 4),
        "CV_F1_%":      round(cv_f1.mean()*100, 2),
        "CV_F1_Std_%":  round(cv_f1.std()*100, 2),
        "_yp": yp, "_yprob": yprob,
        "_cm": confusion_matrix(y_test,yp)
    }
    print(f"\n    [{name}]")
    for k,v in result.items():
        if not k.startswith("_"):
            print(f"      {k:15s}: {v}")
    return result

r1 = evaluate(model1, m1_name)
r2 = evaluate(model2, m2_name)

# ── STEP 6: Per-Sector Evaluation ────────────────────────────
print("\n[6] Per-sector accuracy...")
sec_rows = []
for sec in ["Banking","IT","Pharma"]:
    mask = (test_df["sector"] == sec).values
    Xs, ys = X_test_sc[mask], y_test[mask]
    for model, name in [(model1,m1_name),(model2,m2_name)]:
        yp   = model.predict(Xs)
        yprb = model.predict_proba(Xs)[:,1]
        row  = {"Sector":sec,"Model":name,
                "N":len(ys),
                "Accuracy_%": round(accuracy_score(ys,yp)*100,2),
                "F1_%":       round(f1_score(ys,yp,zero_division=0)*100,2),
                "ROC_AUC":    round(roc_auc_score(ys,yprb),4)}
        sec_rows.append(row)
        print(f"    {sec:10s} {name:20s} Acc:{row['Accuracy_%']}% AUC:{row['ROC_AUC']}")

# ── STEP 7: Feature Importance ───────────────────────────────
print("\n[7] Feature importance...")
fi_df = pd.DataFrame({
    "Feature":  FEATURE_COLS,
    m1_name:    model1.feature_importances_,
    m2_name:    model2.feature_importances_
}).sort_values(m1_name, ascending=False)
print(fi_df[["Feature",m1_name,m2_name]].head(5).to_string(index=False))

# ── STEP 8: Save CSVs ─────────────────────────────────────────
print("\n[8] Saving CSVs...")
full_df[["Date","ticker","sector"]+FEATURE_COLS+["target"]].to_csv(
    "task6_feature_matrix.csv", index=False)
save_cols = ["Model","Accuracy_%","Precision_%","Recall_%","F1_%","ROC_AUC","CV_F1_%","CV_F1_Std_%"]
pd.DataFrame([{k:v for k,v in r.items() if k in save_cols} for r in [r1,r2]]
    ).to_csv("task6_model_comparison.csv", index=False)
pd.DataFrame(sec_rows).to_csv("task6_sector_eval.csv", index=False)
fi_df.to_csv("task6_feature_importance.csv", index=False)

# ── STEP 9: Charts ────────────────────────────────────────────
print("\n[9] Generating charts...")
DARK, SURF = "#0f1117", "#1a1a2e"
BLU,  GRN  = "#42a5f5", "#66bb6a"
WH         = "#e0e0e0"

# Feature Importance
fig, ax = plt.subplots(figsize=(12,6))
fig.patch.set_facecolor(DARK); ax.set_facecolor(SURF)
top = fi_df.head(10); x = np.arange(len(top)); w = 0.35
ax.bar(x-w/2, top[m1_name], w, color=BLU,  label=m1_name, zorder=3)
ax.bar(x+w/2, top[m2_name], w, color=GRN, label=m2_name, zorder=3)
ax.set_xticks(x)
ax.set_xticklabels(top["Feature"], rotation=35, ha="right", color=WH, fontsize=10)
ax.set_title("Top 10 Feature Importances — Task 6", color=WH, fontweight="bold", fontsize=13)
ax.set_ylabel("Importance", color=WH)
ax.tick_params(colors=WH)
ax.legend(labelcolor=WH, facecolor="#222", edgecolor="#555")
ax.grid(True, axis="y", color="#2a2a3e", lw=0.5)
for sp in ["top","right"]: ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig("task6_feature_importance.png", dpi=150, facecolor=DARK)
plt.close()

# ROC Curves
fig, ax = plt.subplots(figsize=(7,6))
fig.patch.set_facecolor(DARK); ax.set_facecolor(SURF)
for res, color, name in [(r1,BLU,m1_name),(r2,GRN,m2_name)]:
    fpr, tpr, _ = roc_curve(y_test, res["_yprob"])
    ax.plot(fpr, tpr, color=color, lw=2.5,
            label=f"{name} AUC={res['ROC_AUC']}")
ax.plot([0,1],[0,1], color="#555", lw=1.5, ls="--", label="Random 0.500")
ax.set_xlabel("False Positive Rate", color=WH)
ax.set_ylabel("True Positive Rate", color=WH)
ax.set_title("ROC Curves — Outperformance Classifier", color=WH, fontweight="bold")
ax.tick_params(colors=WH)
ax.legend(labelcolor=WH, facecolor="#222", edgecolor="#555")
ax.grid(True, color="#2a2a3e", lw=0.5)
plt.tight_layout()
plt.savefig("task6_roc_curves.png", dpi=150, facecolor=DARK)
plt.close()

# Confusion Matrices
fig, axes = plt.subplots(1,2, figsize=(12,5))
fig.patch.set_facecolor(DARK)
for ax, res, name in [(axes[0],r1,m1_name),(axes[1],r2,m2_name)]:
    ax.set_facecolor(SURF)
    sns.heatmap(res["_cm"], annot=True, fmt="d", cmap="Blues",
                xticklabels=["Under","Outperform"],
                yticklabels=["Under","Outperform"],
                ax=ax, cbar=False, linewidths=0.5,
                annot_kws={"size":14,"color":"white","weight":"bold"})
    ax.set_title(f"{name}\nAcc:{res['Accuracy_%']}%  F1:{res['F1_%']}%  AUC:{res['ROC_AUC']}",
                 color=WH, fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted", color=WH); ax.set_ylabel("Actual", color=WH)
    ax.tick_params(colors=WH)
plt.suptitle("Confusion Matrices — Task 6", color=WH, fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("task6_confusion_matrices.png", dpi=150, facecolor=DARK)
plt.close()

print("\n" + "="*48)
print("ALL TASK 6 FILES SAVED:")
print("  task6_feature_matrix.csv      ✅")
print("  task6_model_comparison.csv    ✅")
print("  task6_sector_eval.csv         ✅")
print("  task6_feature_importance.csv  ✅")
print("  task6_feature_importance.png  ✅")
print("  task6_roc_curves.png          ✅")
print("  task6_confusion_matrices.png  ✅")
print("="*48)

XGBoost ✅

[1] Loading data...
    491 rows | 2023-01-02 to 2024-12-31

[2] Engineering 17 features per stock per day...
    Total samples : 4380
    Outperform %  : 52.8%

[3] Time-series split 80/20...
    Train: 3504 | Test: 876

[4] Training models...
    XGBoost ✅  |  RandomForest ✅

[5] Evaluating on test set...

    [XGBoost]
      Model          : XGBoost
      Accuracy_%     : 46.69
      Precision_%    : 43.4
      Recall_%       : 44.77
      F1_%           : 44.07
      ROC_AUC        : 0.4955
      CV_F1_%        : 55.65
      CV_F1_Std_%    : 9.04

    [RandomForest]
      Model          : RandomForest
      Accuracy_%     : 52.28
      Precision_%    : 48.99
      Recall_%       : 41.36
      F1_%           : 44.85
      ROC_AUC        : 0.5382
      CV_F1_%        : 53.0
      CV_F1_Std_%    : 8.81

[6] Per-sector accuracy...
    Banking    XGBoost              Acc:40.69% AUC:0.4087
    Banking    RandomForest         Acc:45.17% AUC:0.4205
    IT         XGBoost        